# Baselines: SwinIR / EDSR / SRResNet / SGNet / DORNet

Comparative baseline training + evaluation. All baselines use the same
training data (`data/dataset/{train,val}`) and validation set for fair comparison.

Checkpoints written to `checkpoints/baseline_{swinir,edsr,srresnet,sgnet}/`.

**Priority**: SwinIR first (~11h), SGNet second (~4h), DORNet zero-shot (minutes).
EDSR and SRResNet added as classic SR baselines (~3–4h each).


## 0. Setup


In [ ]:
# Shared setup - imports, paths, dataset class.
# Run this cell once per kernel session before any baseline.
from pathlib import Path
import os, sys, time, json, subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "FaceLift" else Path.cwd().resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

DATASET_DIR = PROJECT_ROOT / "data" / "dataset"
CKPT_ROOT   = PROJECT_ROOT / "checkpoints"

# Reuse the canonical dataset from our main trainer so all baselines see exactly
# the same splits + same normalization + same mask handling.
from train_depth_upres import DepthUpResDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Quick sanity check on dataset
train_ds = DepthUpResDataset(DATASET_DIR, "train", lr_kind="8bit")
val_ds   = DepthUpResDataset(DATASET_DIR, "val",   lr_kind="8bit")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

# Check baselines-specific deps (SwinIR needs timm + einops)
_missing = []
for _pkg in ("timm", "einops"):
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"[DEP CHECK] Missing packages for SwinIR/SGNet: {_missing}")
    print(f"  Install with:  {sys.executable} -m pip install " + " ".join(_missing))
    print(f"  Then restart kernel and re-run this cell.")
else:
    print("[DEP CHECK] timm, einops OK")


## 1. SwinIR-lightweight baseline

SwinIR (Liang et al. 2021), lightweight variant (~0.5M params), designed for ×4 SR.
Source: https://github.com/JingyunLiang/SwinIR

Modified: `in_chans=3` → `in_chans=1` (depth only). `upscale=4` natively supported.
This baseline tests whether a transformer-based SR model can learn our degradation
without the bicubic residual prior that our UNet uses.


In [ ]:
# 1.1 Install SwinIR (run once)
SWINIR_DIR = PROJECT_ROOT / "external" / "SwinIR"
if not SWINIR_DIR.exists():
    SWINIR_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone",
                    "https://github.com/JingyunLiang/SwinIR",
                    str(SWINIR_DIR)], check=True)
    print(f"Cloned SwinIR to {SWINIR_DIR}")
else:
    print(f"[SKIP] {SWINIR_DIR} already exists")

# Make SwinIR importable
if str(SWINIR_DIR) not in sys.path:
    sys.path.insert(0, str(SWINIR_DIR))

# Import the model class
from models.network_swinir import SwinIR as SwinIR_Net
print("SwinIR import OK")


In [ ]:
# 1.2 SwinIR-lightweight training
# ~800K params, upscale=4, in_chans=1 for depth.
# TODO: fill in training loop below, mirroring scripts/train_depth_upres.py.

from train_depth_upres import _masked_l1, gradient_loss   # reuse losses

EPOCHS = 150
BATCH  = 1
GRAD_ACC = 8
LR     = 1e-4

ckpt_dir = CKPT_ROOT / "baseline_swinir"
ckpt_dir.mkdir(parents=True, exist_ok=True)

if (ckpt_dir / "best.pt").exists():
    print(f"[SKIP] {ckpt_dir / 'best.pt'} already trained.")
else:
    # --- Model ---
    # LIGHTWEIGHT config tuned for 4070 Laptop 8GB VRAM + 4×SR at 1024×1024:
    #   - 2 RSTB blocks × 4 STL each = 8 attention layers (vs original 36)
    #   - embed_dim = 48 (vs original 60)
    #   - num_heads = 4 (vs 6)
    # Target: ~0.5M params, ~5-8 min/epoch, 150 epoch ≈ 15-20 hours.
    # This is still a legitimate SwinIR baseline — smaller but same architecture.
    model = SwinIR_Net(
        upscale=4, in_chans=1, img_size=64, window_size=8,
        img_range=1.0, depths=[4, 4], embed_dim=48,
        num_heads=[4, 4], mlp_ratio=2, upsampler="pixelshuffledirect",
        resi_connection="1conv",
    ).to(device)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"SwinIR-tiny (lightweight): {n/1e6:.2f}M params")

    # --- Loader ---
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                              num_workers=0, pin_memory=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler()

    history = []; best_val = float("inf")
    for epoch in range(1, EPOCHS + 1):
        model.train(); t0 = time.time(); train_l = 0.0
        optimizer.zero_grad()
        for step, (lr_in, hr, mask) in enumerate(train_loader):
            lr_in, hr, mask = [t.to(device, non_blocking=True) for t in (lr_in, hr, mask)]
            with torch.cuda.amp.autocast():
                pred = model(lr_in).clamp(0, 1)
                l1   = _masked_l1(pred, hr, mask)
                gl   = gradient_loss(pred, hr, mask)
                loss = (l1 + 0.5 * gl) / GRAD_ACC
            scaler.scale(loss).backward()
            if (step + 1) % GRAD_ACC == 0:
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            train_l += loss.item() * GRAD_ACC
        train_l /= len(train_ds)

        # --- Val ---
        model.eval(); val_l = 0.0; val_l1 = 0.0
        with torch.no_grad():
            for lr_in, hr, mask in val_loader:
                lr_in, hr, mask = [t.to(device, non_blocking=True) for t in (lr_in, hr, mask)]
                pred = model(lr_in).clamp(0, 1)
                vl = _masked_l1(pred, hr, mask) + 0.5 * gradient_loss(pred, hr, mask)
                val_l  += vl.item()
                val_l1 += _masked_l1(pred.float(), hr, mask).item()
        val_l  /= len(val_ds); val_l1 /= len(val_ds)

        scheduler.step()
        history.append({"epoch": epoch, "train": train_l, "val": val_l, "val_l1": val_l1,
                        "lr": optimizer.param_groups[0]["lr"], "time_s": time.time()-t0})
        print(f"[{epoch:03d}/{EPOCHS}] train={train_l:.5f} val={val_l:.5f} "
              f"val_l1={val_l1:.5f} time={time.time()-t0:.0f}s")
        if val_l < best_val:
            best_val = val_l
            torch.save({"model": model.state_dict(), "epoch": epoch,
                        "val_loss": val_l, "args": {"base_ch": 60, "arch": "SwinIR-light"}},
                       ckpt_dir / "best.pt")
        (ckpt_dir / "train_log.json").write_text(json.dumps(history, indent=2))
    print(f"Done. Best val: {best_val:.5f}")


## 1b. EDSR baseline (CVPRW 2017)

EDSR (Lim et al. CVPRW 2017) — classic SR baseline used in virtually all SR papers.
Core innovation: residual blocks without batch normalization.

Lightweight config: 16 blocks, 64 channels, ~1.5M params.
Uses pixel shuffle ×4 upsampling + bicubic skip connection.
Defined inline (no external repo needed).


In [ ]:
# === 1b. EDSR-light baseline (CVPRW 2017) ===
# No external repo needed — model defined inline.
from train_depth_upres import _masked_l1, gradient_loss

class ResBlock(nn.Module):
    def __init__(self, n_feats, res_scale=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feats, n_feats, 3, padding=1),
        )
        self.res_scale = res_scale
    def forward(self, x):
        return x + self.body(x) * self.res_scale

class EDSR_Light(nn.Module):
    """EDSR-baseline: 16 ResBlocks, 64 channels, no BN, ×4 pixel shuffle."""
    def __init__(self, n_feats=64, n_blocks=16, in_ch=1, scale=4):
        super().__init__()
        self.head = nn.Conv2d(in_ch, n_feats, 3, padding=1)
        self.body = nn.Sequential(*[ResBlock(n_feats) for _ in range(n_blocks)])
        self.body_tail = nn.Conv2d(n_feats, n_feats, 3, padding=1)
        # Pixel shuffle upsampler: ×4 = two ×2 stages
        self.up = nn.Sequential(
            nn.Conv2d(n_feats, n_feats * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.Conv2d(n_feats, n_feats * 4, 3, padding=1),
            nn.PixelShuffle(2),
        )
        self.tail = nn.Conv2d(n_feats, 1, 3, padding=1)
    def forward(self, x):
        h = self.head(x)
        h = h + self.body_tail(self.body(h))
        h = self.up(h)
        # Bicubic skip connection (same as our UNet for fair comparison)
        bic = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False).clamp(0, 1)
        return (bic + self.tail(h) * 0.5).clamp(0, 1)

EDSR_EPOCHS = 100
edsr_ckpt_dir = CKPT_ROOT / 'baseline_edsr'
edsr_ckpt_dir.mkdir(parents=True, exist_ok=True)

if (edsr_ckpt_dir / 'best.pt').exists():
    print(f'[SKIP] EDSR already trained: {edsr_ckpt_dir / "best.pt"}')
else:
    model = EDSR_Light(n_feats=64, n_blocks=16, in_ch=1, scale=4).to(device)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'EDSR-light: {n/1e6:.2f}M params')

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                            num_workers=0, pin_memory=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EDSR_EPOCHS)
    scaler = torch.cuda.amp.GradScaler()

    history = []; best_val_l1 = float('inf')
    for epoch in range(1, EDSR_EPOCHS + 1):
        model.train(); t0 = time.time(); train_l = 0.0
        optimizer.zero_grad()
        for step, (lr_in, hr, mask) in enumerate(train_loader):
            lr_in, hr, mask = [t.to(device) for t in (lr_in, hr, mask)]
            with torch.cuda.amp.autocast():
                pred = model(lr_in).clamp(0, 1)
                loss = (_masked_l1(pred, hr, mask) + 0.5 * gradient_loss(pred, hr, mask)) / 8
            scaler.scale(loss).backward()
            if (step + 1) % 8 == 0:
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            train_l += loss.item() * 8
        train_l /= len(train_ds)

        model.eval(); val_l1 = 0.0
        with torch.no_grad():
            for lr_in, hr, mask in val_loader:
                lr_in, hr, mask = [t.to(device) for t in (lr_in, hr, mask)]
                pred = model(lr_in).clamp(0, 1)
                val_l1 += _masked_l1(pred.float(), hr, mask).item()
        val_l1 /= len(val_ds)
        scheduler.step()

        history.append({'epoch': epoch, 'train_loss': train_l, 'val_l1': val_l1,
                        'lr': optimizer.param_groups[0]['lr'], 'time_s': time.time()-t0})
        print(f'[{epoch:03d}/{EDSR_EPOCHS}] train={train_l:.5f} val_l1={val_l1:.5f} '
              f'time={time.time()-t0:.0f}s')
        if val_l1 < best_val_l1:
            best_val_l1 = val_l1
            torch.save({'model': model.state_dict(), 'epoch': epoch,
                        'val_l1': val_l1, 'args': {'arch': 'EDSR-light', 'n_feats': 64}},
                       edsr_ckpt_dir / 'best.pt')
            print(f'    -> saved best.pt (val_l1={val_l1:.5f})')
        (edsr_ckpt_dir / 'train_log.json').write_text(json.dumps(history, indent=2))
    print(f'Done. Best val_l1: {best_val_l1:.5f}')


## 1c. SRResNet baseline (CVPR 2017)

SRResNet (Ledig et al. CVPR 2017) — generator part of SRGAN.
Unlike EDSR, uses batch normalization in residual blocks.
Comparison shows BN's impact on depth SR performance.

Same config: 16 blocks, 64 channels, ~1.5M params, pixel shuffle ×4.


In [ ]:
# === 1c. SRResNet baseline (CVPR 2017) ===
from train_depth_upres import _masked_l1, gradient_loss

class ResBlockBN(nn.Module):
    def __init__(self, n_feats):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, padding=1),
            nn.BatchNorm2d(n_feats),
            nn.PReLU(),
            nn.Conv2d(n_feats, n_feats, 3, padding=1),
            nn.BatchNorm2d(n_feats),
        )
    def forward(self, x):
        return x + self.body(x)

class SRResNet(nn.Module):
    """SRResNet: 16 ResBlocks with BN, PReLU, pixel shuffle ×4."""
    def __init__(self, n_feats=64, n_blocks=16, in_ch=1, scale=4):
        super().__init__()
        self.head = nn.Sequential(
            nn.Conv2d(in_ch, n_feats, 9, padding=4),
            nn.PReLU(),
        )
        self.body = nn.Sequential(*[ResBlockBN(n_feats) for _ in range(n_blocks)])
        self.body_tail = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, padding=1),
            nn.BatchNorm2d(n_feats),
        )
        self.up = nn.Sequential(
            nn.Conv2d(n_feats, n_feats * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
            nn.Conv2d(n_feats, n_feats * 4, 3, padding=1),
            nn.PixelShuffle(2),
            nn.PReLU(),
        )
        self.tail = nn.Conv2d(n_feats, 1, 9, padding=4)
    def forward(self, x):
        h = self.head(x)
        h = h + self.body_tail(self.body(h))
        h = self.up(h)
        # Bicubic skip connection
        bic = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False).clamp(0, 1)
        return (bic + self.tail(h) * 0.5).clamp(0, 1)

SRRESNET_EPOCHS = 100
srresnet_ckpt_dir = CKPT_ROOT / 'baseline_srresnet'
srresnet_ckpt_dir.mkdir(parents=True, exist_ok=True)

if (srresnet_ckpt_dir / 'best.pt').exists():
    print(f'[SKIP] SRResNet already trained: {srresnet_ckpt_dir / "best.pt"}')
else:
    model = SRResNet(n_feats=64, n_blocks=16, in_ch=1, scale=4).to(device)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'SRResNet: {n/1e6:.2f}M params')

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                              num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False,
                            num_workers=0, pin_memory=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SRRESNET_EPOCHS)
    scaler = torch.cuda.amp.GradScaler()

    history = []; best_val_l1 = float('inf')
    for epoch in range(1, SRRESNET_EPOCHS + 1):
        model.train(); t0 = time.time(); train_l = 0.0
        optimizer.zero_grad()
        for step, (lr_in, hr, mask) in enumerate(train_loader):
            lr_in, hr, mask = [t.to(device) for t in (lr_in, hr, mask)]
            with torch.cuda.amp.autocast():
                pred = model(lr_in).clamp(0, 1)
                loss = (_masked_l1(pred, hr, mask) + 0.5 * gradient_loss(pred, hr, mask)) / 8
            scaler.scale(loss).backward()
            if (step + 1) % 8 == 0:
                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            train_l += loss.item() * 8
        train_l /= len(train_ds)

        model.eval(); val_l1 = 0.0
        with torch.no_grad():
            for lr_in, hr, mask in val_loader:
                lr_in, hr, mask = [t.to(device) for t in (lr_in, hr, mask)]
                pred = model(lr_in).clamp(0, 1)
                val_l1 += _masked_l1(pred.float(), hr, mask).item()
        val_l1 /= len(val_ds)
        scheduler.step()

        history.append({'epoch': epoch, 'train_loss': train_l, 'val_l1': val_l1,
                        'lr': optimizer.param_groups[0]['lr'], 'time_s': time.time()-t0})
        print(f'[{epoch:03d}/{SRRESNET_EPOCHS}] train={train_l:.5f} val_l1={val_l1:.5f} '
              f'time={time.time()-t0:.0f}s')
        if val_l1 < best_val_l1:
            best_val_l1 = val_l1
            torch.save({'model': model.state_dict(), 'epoch': epoch,
                        'val_l1': val_l1, 'args': {'arch': 'SRResNet', 'n_feats': 64}},
                       srresnet_ckpt_dir / 'best.pt')
            print(f'    -> saved best.pt (val_l1={val_l1:.5f})')
        (srresnet_ckpt_dir / 'train_log.json').write_text(json.dumps(history, indent=2))
    print(f'Done. Best val_l1: {best_val_l1:.5f}')


## 2. SGNet baseline (AAAI 2024)

SGNet (Yan et al. AAAI 2024) — RGB-guided depth SR with frequency-domain loss.
Source: https://github.com/yanzq95/SGNet

Takes (RGB, LR_depth) → HR_depth. `num_feats=40` (original), ~25M params.
Trained patch-based (256×256 HR, 64×64 LR) to fit 8GB VRAM.
Validation at 512×512 (full 1024 OOMs with RGB guidance).


In [ ]:
# 2.1 SGNet training (AAAI 2024, RGB-guided depth SR)
# Patch-based: 128×128 HR patches, 32×32 LR. Val center-crop 128.
# num_feats=24 (~9M params). NO AMP — SGNet FFT kills FP16 gradients.
# --resume: continues from best.pt if training was interrupted.
import subprocess

sgnet_ckpt = CKPT_ROOT / 'baseline_sgnet' / 'best.pt'
sgnet_log  = CKPT_ROOT / 'baseline_sgnet' / 'train_log.json'

# Check if already fully trained
need_train = True
if sgnet_log.exists():
    import json as _json
    _log = _json.loads(sgnet_log.read_text())
    if len(_log) >= 100:
        best = min(_log, key=lambda x: x['val_l1'])
        print(f'[SKIP] SGNet fully trained ({len(_log)} epochs).')
        print(f"  best val_l1={best['val_l1']:.5f} @ epoch {best['epoch']}")
        need_train = False

if need_train:
    cmd = [
        sys.executable, 'scripts/train_sgnet.py',
        '--epochs', '100',
        '--batch_size', '4',
        '--grad_accum', '2',
        '--patch_size', '128',
        '--num_feats', '24',
        '--no_amp',
        '--resume',
        '--checkpoints', str(CKPT_ROOT / 'baseline_sgnet'),
    ]
    print('CMD:', ' '.join(cmd))
    print('-' * 60)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, encoding='utf-8', errors='replace')
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print(f'\nSGNet exit code: {proc.returncode}')


## 3. DORNet baseline (CVPR 2025 oral) — Zero-shot

DORNet (CVPR 2025 oral) — degradation-oriented MoE + deformable conv.
Paper: arXiv 2410.11666. Source: https://github.com/yanzq95/DORNet

**Why zero-shot (not retrained)**:
1. mmcv modulated deform conv requires CUDA compilation, painful on Windows
2. ContrastLoss uses VGG19 (3-ch) — depth repeat adds VRAM overhead
3. 300 epochs at 1024×1024 would likely OOM on 8GB

Uses NYU-v2 ×4 pretrained weights. NYU assumes bicubic downsampling;
our data uses 3DGS render + 8-bit quantization. A poor result here
directly supports claim (a): rendering degradation ≠ standard degradation.

Implementation: `torchvision.ops.deform_conv2d` instead of mmcv (no extra install).


In [ ]:
# 3.1 DORNet zero-shot evaluation (CVPR 2025 oral)
# Uses pretrained NYU_X4.pth weights -- no training needed.
# Tests cross-domain generalization: NYU bicubic -> 3DGS rendering degradation.
import subprocess, json as _json

dornet_result = PROJECT_ROOT / 'eval' / 'dornet_zeroshot_results.json'

if dornet_result.exists():
    with open(dornet_result) as f:
        res = _json.load(f)
    print('[SKIP] DORNet zero-shot already evaluated')
    print('  val_l1  = %.5f' % res['val_l1'])
    print('  val_PSNR = %.2f dB' % res['val_psnr'])
else:
    cmd = [
        sys.executable, 'scripts/eval_dornet_zeroshot.py',
        '--resolution', '1024',
        '--amp',
    ]
    print('CMD:', ' '.join(cmd))
    print('Expected: ~5-10 min (inference only, no training)')
    print('-' * 60)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, encoding='utf-8', errors='replace')
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    print('DORNet exit code:', proc.returncode)
    if proc.returncode != 0:
        print('If OOM, change resolution to 512 and re-run')


## 4. Collect all results + merge into final table

Scans `checkpoints/baseline_*/best.pt`, evaluates on val set,
outputs merged table (our UNet + all baselines).
Saved to `eval/baseline_full_table.csv`.


In [ ]:
# 4. Eval all methods on val set — unified table
# Loads each model, runs inference on val, computes L1/RMSE/PSNR.
# Includes: UNet variants + EDSR + SRResNet + SwinIR + DORNet(zero-shot) + Bicubic-only
import glob, json as _json
from PIL import Image
import cv2
from train_depth_upres import DepthUpResDataset, _masked_l1, gradient_loss
from train_depth_upres import DepthUpResUNet

val_ds = DepthUpResDataset(DATASET_DIR, 'val', lr_kind='8bit')
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)

def eval_model(model, loader, device):
    model.eval()
    total_l1 = 0.0; total_mse = 0.0; n = 0
    with torch.no_grad():
        for lr_in, hr, mask in loader:
            lr_in, hr, mask = lr_in.to(device), hr.to(device), mask.to(device)
            with torch.cuda.amp.autocast():
                out = model(lr_in)
                if isinstance(out, tuple): out = out[0]
                out = out.clamp(0, 1)
            diff = (out.float() - hr).abs() * mask
            total_l1 += (diff.sum() / mask.sum().clamp(min=1e-6)).item()
            total_mse += (((out.float()-hr)**2 * mask).sum() / mask.sum().clamp(min=1e-6)).item()
            n += 1
    l1 = total_l1 / n
    mse = total_mse / n
    psnr = -10 * np.log10(mse + 1e-12)
    rmse = np.sqrt(mse)
    return {'L1': l1, 'RMSE': rmse, 'PSNR': psnr}

rows = []

# --- 1. Bicubic-only baseline (no learned model) ---
print('Evaluating: Bicubic-only...')
bic_l1 = 0.0; bic_mse = 0.0; n = 0
with torch.no_grad():
    for lr_in, hr, mask in val_loader:
        bic = F.interpolate(lr_in, scale_factor=4, mode='bicubic', align_corners=False).clamp(0,1)
        bic, hr, mask = bic.to(device), hr.to(device), mask.to(device)
        diff = (bic - hr).abs() * mask
        bic_l1 += (diff.sum() / mask.sum().clamp(min=1e-6)).item()
        bic_mse += (((bic-hr)**2 * mask).sum() / mask.sum().clamp(min=1e-6)).item()
        n += 1
bic_l1 /= n; bic_mse /= n
rows.append({'Method': 'Bicubic (no model)', 'Params': '-', 'Training': '-',
             'L1': bic_l1, 'RMSE': np.sqrt(bic_mse),
             'PSNR': -10*np.log10(bic_mse+1e-12)})
print('  L1=%.5f PSNR=%.2f' % (bic_l1, rows[-1]['PSNR']))

# --- 2. UNet (ours, 8-bit) ---
unet_ckpt = CKPT_ROOT / 'depth_upres_8bit_ch32' / 'best.pt'
if unet_ckpt.exists():
    print('Evaluating: UNet (8-bit)...')
    model = DepthUpResUNet(base_ch=32, predict_normal=False).to(device)
    ckpt = torch.load(unet_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'], strict=False)
    m = eval_model(model, val_loader, device)
    rows.append({'Method': 'UNet (ours)', 'Params': '7.77M', 'Training': '3DGS',
                 'L1': m['L1'], 'RMSE': m['RMSE'], 'PSNR': m['PSNR']})
    print('  L1=%.5f PSNR=%.2f' % (m['L1'], m['PSNR']))
    del model; torch.cuda.empty_cache()

# --- 3. UNet + Normal (DSINE) ---
unet_n_ckpt = CKPT_ROOT / 'depth_upres_8bit_ch32_normal_w050_dsine' / 'best.pt'
if unet_n_ckpt.exists():
    print('Evaluating: UNet+Normal (DSINE)...')
    model = DepthUpResUNet(base_ch=32, predict_normal=True).to(device)
    ckpt = torch.load(unet_n_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'], strict=False)
    m = eval_model(model, val_loader, device)
    rows.append({'Method': 'UNet+Normal (ours)', 'Params': '7.78M', 'Training': '3DGS',
                 'L1': m['L1'], 'RMSE': m['RMSE'], 'PSNR': m['PSNR']})
    print('  L1=%.5f PSNR=%.2f' % (m['L1'], m['PSNR']))
    del model; torch.cuda.empty_cache()

# --- 4. EDSR ---
edsr_ckpt = CKPT_ROOT / 'baseline_edsr' / 'best.pt'
if edsr_ckpt.exists():
    print('Evaluating: EDSR...')
    # Re-define EDSR inline (same as training cell)
    class _ResBlock(nn.Module):
        def __init__(self, n, s=0.1):
            super().__init__()
            self.body = nn.Sequential(nn.Conv2d(n,n,3,padding=1), nn.ReLU(True), nn.Conv2d(n,n,3,padding=1))
            self.res_scale = s
        def forward(self, x): return x + self.body(x) * self.res_scale
    class _EDSR(nn.Module):
        def __init__(self, nf=64, nb=16):
            super().__init__()
            self.head = nn.Conv2d(1, nf, 3, padding=1)
            self.body = nn.Sequential(*[_ResBlock(nf) for _ in range(nb)])
            self.body_tail = nn.Conv2d(nf, nf, 3, padding=1)
            self.up = nn.Sequential(nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2),
                                    nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2))
            self.tail = nn.Conv2d(nf, 1, 3, padding=1)
        def forward(self, x):
            h = self.head(x); h = h + self.body_tail(self.body(h)); h = self.up(h)
            bic = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False).clamp(0,1)
            return (bic + self.tail(h)*0.5).clamp(0,1)
    model = _EDSR().to(device)
    ckpt = torch.load(edsr_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    m = eval_model(model, val_loader, device)
    rows.append({'Method': 'EDSR-light', 'Params': '%.2fM' % n_params, 'Training': '3DGS',
                 'L1': m['L1'], 'RMSE': m['RMSE'], 'PSNR': m['PSNR']})
    print('  L1=%.5f PSNR=%.2f' % (m['L1'], m['PSNR']))
    del model; torch.cuda.empty_cache()

# --- 5. SRResNet ---
srr_ckpt = CKPT_ROOT / 'baseline_srresnet' / 'best.pt'
if srr_ckpt.exists():
    print('Evaluating: SRResNet...')
    class _ResBlockBN(nn.Module):
        def __init__(self, n):
            super().__init__()
            self.body = nn.Sequential(nn.Conv2d(n,n,3,padding=1), nn.BatchNorm2d(n), nn.PReLU(),
                                      nn.Conv2d(n,n,3,padding=1), nn.BatchNorm2d(n))
        def forward(self, x): return x + self.body(x)
    class _SRResNet(nn.Module):
        def __init__(self, nf=64, nb=16):
            super().__init__()
            self.head = nn.Sequential(nn.Conv2d(1, nf, 9, padding=4), nn.PReLU())
            self.body = nn.Sequential(*[_ResBlockBN(nf) for _ in range(nb)])
            self.body_tail = nn.Sequential(nn.Conv2d(nf, nf, 3, padding=1), nn.BatchNorm2d(nf))
            self.up = nn.Sequential(nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2), nn.PReLU(),
                                    nn.Conv2d(nf, nf*4, 3, padding=1), nn.PixelShuffle(2), nn.PReLU())
            self.tail = nn.Conv2d(nf, 1, 9, padding=4)
        def forward(self, x):
            h = self.head(x); h = h + self.body_tail(self.body(h)); h = self.up(h)
            bic = F.interpolate(x, scale_factor=4, mode='bicubic', align_corners=False).clamp(0,1)
            return (bic + self.tail(h)*0.5).clamp(0,1)
    model = _SRResNet().to(device)
    ckpt = torch.load(srr_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    m = eval_model(model, val_loader, device)
    rows.append({'Method': 'SRResNet', 'Params': '%.2fM' % n_params, 'Training': '3DGS',
                 'L1': m['L1'], 'RMSE': m['RMSE'], 'PSNR': m['PSNR']})
    print('  L1=%.5f PSNR=%.2f' % (m['L1'], m['PSNR']))
    del model; torch.cuda.empty_cache()

# --- 6. SwinIR ---
swinir_ckpt = CKPT_ROOT / 'baseline_swinir' / 'best.pt'
if swinir_ckpt.exists():
    print('Evaluating: SwinIR...')
    SWINIR_DIR = PROJECT_ROOT / 'external' / 'SwinIR'
    if str(SWINIR_DIR) not in sys.path: sys.path.insert(0, str(SWINIR_DIR))
    from models.network_swinir import SwinIR as SwinIR_Net
    model = SwinIR_Net(upscale=4, in_chans=1, img_size=64, window_size=8,
                       img_range=1.0, depths=[4,4], embed_dim=48,
                       num_heads=[4,4], mlp_ratio=2, upsampler='pixelshuffledirect',
                       resi_connection='1conv').to(device)
    ckpt = torch.load(swinir_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    m = eval_model(model, val_loader, device)
    rows.append({'Method': 'SwinIR-tiny', 'Params': '%.2fM' % n_params, 'Training': '3DGS',
                 'L1': m['L1'], 'RMSE': m['RMSE'], 'PSNR': m['PSNR']})
    print('  L1=%.5f PSNR=%.2f' % (m['L1'], m['PSNR']))
    del model; torch.cuda.empty_cache()

# --- 7. DORNet zero-shot (from saved results) ---
dornet_json = PROJECT_ROOT / 'eval' / 'dornet_zeroshot_results.json'
if dornet_json.exists():
    print('Loading: DORNet zero-shot results...')
    with open(dornet_json) as f:
        dr = _json.load(f)
    rows.append({'Method': 'DORNet (NYU zero-shot)', 'Params': '3.05M', 'Training': 'NYU bicubic',
                 'L1': dr['val_l1'], 'RMSE': np.sqrt(dr['val_mse']),
                 'PSNR': dr['val_psnr']})
    print('  L1=%.5f PSNR=%.2f' % (dr['val_l1'], dr['val_psnr']))


# --- 8. SGNet (RGB-guided, trained on 3DGS) ---
sgnet_ckpt = CKPT_ROOT / 'baseline_sgnet' / 'best.pt'
if sgnet_ckpt.exists():
    print('Evaluating: SGNet...')
    SGNET_DIR = PROJECT_ROOT / 'external' / 'SGNet'
    if str(SGNET_DIR) not in sys.path: sys.path.insert(0, str(SGNET_DIR))
    from models.SGNet import SGNet as SGNet_Net
    model = SGNet_Net(num_feats=24, kernel_size=3, scale=4).to(device)
    ckpt = torch.load(sgnet_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    # SGNet needs RGB + LR depth. Eval at 512 (1024 OOMs with 25M model).
    # Load images manually for SGNet eval.
    model.eval()
    sg_l1 = 0.0; sg_mse = 0.0; sg_n = 0
    EVAL_SIZE = 512  # HR eval resolution (LR = 128)
    with torch.no_grad():
        for idx in range(len(val_ds)):
            name = val_ds.samples[idx]
            # HR depth
            hr_pil = Image.open(val_ds.hr_dir / f'{name}.png')
            hr_np = np.asarray(hr_pil).astype(np.float32)
            hr_np = hr_np / 65535.0 if hr_np.max() > 255 else hr_np / 255.0
            hr_np = cv2.resize(hr_np, (EVAL_SIZE, EVAL_SIZE), interpolation=cv2.INTER_AREA)
            # LR depth
            lr_pil = Image.open(val_ds.lr_dir / f'{name}.png')
            lr_np = np.asarray(lr_pil).astype(np.float32)
            lr_np = lr_np / 255.0 if lr_np.max() <= 255 else lr_np / 65535.0
            lr_np = cv2.resize(lr_np, (EVAL_SIZE // 4, EVAL_SIZE // 4), interpolation=cv2.INTER_AREA)
            # RGB
            rgb_path = DATASET_DIR / 'val' / 'image' / f'{name}.png'
            rgb_np = np.array(Image.open(rgb_path).convert('RGB').resize((EVAL_SIZE, EVAL_SIZE), Image.BICUBIC), dtype=np.float32) / 255.0
            # Mask
            mask_path = DATASET_DIR / 'val' / 'mask' / f'{name}.png'
            if mask_path.exists():
                m_np = (np.array(Image.open(mask_path).convert('L')) > 127).astype(np.float32)
                m_np = cv2.resize(m_np, (EVAL_SIZE, EVAL_SIZE), interpolation=cv2.INTER_NEAREST)
            else:
                m_np = np.ones((EVAL_SIZE, EVAL_SIZE), dtype=np.float32)
            # To tensors
            hr_t = torch.from_numpy(hr_np).unsqueeze(0).unsqueeze(0).to(device)
            lr_t = torch.from_numpy(lr_np).unsqueeze(0).unsqueeze(0).to(device)
            rgb_t = torch.from_numpy(rgb_np.transpose(2,0,1)).unsqueeze(0).to(device)
            m_t = torch.from_numpy(m_np).unsqueeze(0).unsqueeze(0).to(device)
            out, _ = model((rgb_t, lr_t))
            out = out.clamp(0, 1).float()
            diff = (out - hr_t).abs() * m_t
            sg_l1 += (diff.sum() / m_t.sum().clamp(min=1e-6)).item()
            sg_mse += (((out - hr_t)**2 * m_t).sum() / m_t.sum().clamp(min=1e-6)).item()
            sg_n += 1
    sg_l1 /= sg_n; sg_mse /= sg_n
    rows.append({'Method': 'SGNet (RGB-guided)', 'Params': '%.2fM' % n_params, 'Training': '3DGS',
                 'L1': sg_l1, 'RMSE': np.sqrt(sg_mse),
                 'PSNR': -10*np.log10(sg_mse+1e-12)})
    print('  L1=%.5f PSNR=%.2f (eval@%d)' % (sg_l1, rows[-1]['PSNR'], EVAL_SIZE))
    del model; torch.cuda.empty_cache()

# --- Build table ---
df = pd.DataFrame(rows)
df = df.sort_values('L1')
print('\n' + '='*70)
print('FULL BASELINE TABLE (val set, 129 samples, masked L1)')
print('='*70)
print(df.to_string(index=False, float_format='%.5f'))
print('='*70)

# Save
eval_dir = PROJECT_ROOT / 'eval'
eval_dir.mkdir(exist_ok=True)
csv_path = eval_dir / 'baseline_full_table.csv'
df.to_csv(csv_path, index=False)
print('Saved to:', csv_path)


In [ ]:
# 4.1 Baseline comparison — bar charts + params-vs-PSNR scatter
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm', 'font.size': 10,
    'axes.labelsize': 11, 'axes.titlesize': 12,
    'xtick.labelsize': 9, 'ytick.labelsize': 9.5, 'legend.fontsize': 8.5,
    'figure.dpi': 200,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'grid.linewidth': 0.4, 'grid.alpha': 0.25,
})

df_sorted = df.sort_values('L1')
names = df_sorted['Method'].tolist()
l1_vals = df_sorted['L1'].tolist()
psnr_vals = df_sorted['PSNR'].tolist()

def _parse_params(p):
    if isinstance(p, str) and 'M' in p: return float(p.replace('M', ''))
    try: return float(p)
    except: return None
params_vals = [_parse_params(p) for p in df_sorted['Params'].tolist()]
training_vals = df_sorted['Training'].tolist()

C_OURS = '#1a56db'; C_3DGS = '#4a9ec5'; C_NYU = '#dc2626'; C_NONE = '#9ca3af'
def _color(m, t):
    if 'ours' in str(m): return C_OURS
    if t == '3DGS': return C_3DGS
    if 'NYU' in str(t): return C_NYU
    return C_NONE
colors = [_color(m, t) for m, t in zip(names, training_vals)]
fills  = [c + '25' for c in colors]

# ---- Figure 1: L1 + PSNR bars ----
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'wspace': 0.38})
y = np.arange(len(names))
for ax_i, (vals, label, title, fmt, best_fn) in enumerate([
    (l1_vals, r'Masked L$_1$ $\downarrow$', r'(a) L$_1$ error', '.5f', min),
    (psnr_vals, r'PSNR (dB) $\uparrow$', '(b) PSNR', '.1f', max),
]):
    ax = axes[ax_i]
    bars = ax.barh(y, vals, height=0.56, color=fills, edgecolor=colors, linewidth=1.1)
    ax.set_yticks(y); ax.set_yticklabels(names, fontsize=9)
    ax.invert_yaxis(); ax.set_xlabel(label)
    ax.set_title(title, fontweight='bold', pad=8, fontsize=11)
    if ax_i == 0: ax.set_xscale('log')
    else: ax.set_xlim(0, max(vals) * 1.15)
    ax.grid(axis='x', linestyle=':', color='#999', alpha=0.3)
    best = best_fn(vals)
    for i, (bar, v) in enumerate(zip(bars, vals)):
        w = 'bold' if v == best else 'normal'
        c = colors[i] if v == best else '#777'
        offset = v * 1.15 if ax_i == 0 else v + 0.3
        ax.text(offset, i, f'{v:{fmt}}', va='center', fontsize=7.5, fontweight=w, color=c)
leg = [Patch(facecolor=C_OURS+'25', edgecolor=C_OURS, lw=1.1, label='Ours'),
       Patch(facecolor=C_3DGS+'25', edgecolor=C_3DGS, lw=1.1, label='Baseline (3DGS-trained)'),
       Patch(facecolor=C_NYU+'25',  edgecolor=C_NYU,  lw=1.1, label='DORNet (NYU, zero-shot)'),
       Patch(facecolor=C_NONE+'25', edgecolor=C_NONE,  lw=1.1, label='Bicubic')]
fig.legend(handles=leg, loc='lower center', ncol=4, frameon=False,
           bbox_to_anchor=(0.5, -0.03), fontsize=8)
plt.tight_layout(); plt.subplots_adjust(bottom=0.15)
eval_dir = PROJECT_ROOT / 'eval'; eval_dir.mkdir(exist_ok=True)
fig.savefig(eval_dir / 'fig_baseline_bars.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(eval_dir / 'fig_baseline_bars.pdf', bbox_inches='tight', facecolor='white')
print('Saved fig_baseline_bars')
plt.show()

# ---- Figure 2: Params vs PSNR scatter ----
fig2, ax2 = plt.subplots(figsize=(6.5, 4.5))
sd = [(n, p, ps, _color(n, t)) for n, p, ps, t in zip(names, params_vals, psnr_vals, training_vals)
      if p is not None and p > 0]

for n, p, ps, c in sd:
    ours = 'ours' in n
    ax2.scatter(p, ps, s=55 if ours else 35, c=c+'35', edgecolors=c,
                linewidth=1.4 if ours else 1.0, marker='D' if ours else 'o',
                zorder=10 if ours else 5)

# Simple text labels with careful per-point placement (no arrows)
for n, p, ps, c in sd:
    dn = n.replace(' (ours)', '').replace(' (NYU zero-shot)', '')
    if 'SGNet' in n: dn = r'SGNet$^\dagger$'
    fw = 'bold' if 'ours' in n else 'normal'
    fs = 8 if 'ours' in n else 7.5
    if 'SRResNet' in n:
        ax2.text(p, ps + 0.8, dn, fontsize=fs, color=c, fontweight=fw, ha='center', va='bottom')
    elif 'UNet+Normal' in n:
        ax2.text(p + 0.8, ps + 0.8, dn, fontsize=fs, color=c, fontweight=fw, ha='left', va='bottom')
    elif n == 'UNet (ours)':
        ax2.text(p + 0.8, ps - 0.5, dn, fontsize=fs, color=c, fontweight=fw, ha='left', va='top')
    elif 'EDSR' in n:
        ax2.text(p, ps - 0.8, dn, fontsize=fs, color=c, fontweight=fw, ha='center', va='top')
    elif 'DORNet' in n:
        ax2.text(p + 0.5, ps - 0.8, dn, fontsize=fs, color=c, fontweight=fw, ha='left', va='top')
    elif 'SGNet' in n:
        ax2.text(p, ps - 0.8, dn, fontsize=fs, color=c, fontweight=fw, ha='center', va='top')
    elif 'SwinIR' in n:
        ax2.text(p + 0.06, ps + 0.8, dn, fontsize=fs, color=c, fontweight=fw, ha='left', va='bottom')

ax2.set_xscale('log'); ax2.set_xlim(0.15, 15); ax2.set_ylim(9, 50)
ax2.set_xlabel('Parameters (M)'); ax2.set_ylabel('PSNR (dB)')
ax2.set_title('(c) Efficiency vs. quality', fontweight='bold', fontsize=11, pad=10)
ax2.grid(True, linestyle=':', alpha=0.2)
ax2.annotate('', xy=(0.18, 49.5), xytext=(0.4, 47),
             arrowprops=dict(arrowstyle='->', color='#d0d0d0', lw=1.0))
ax2.text(0.16, 49.8, 'ideal', fontsize=7, color='#c0c0c0', fontstyle='italic')
ax2.text(0.02, 0.02, r'$\dagger$ eval@512 (1024 OOM)', transform=ax2.transAxes,
         fontsize=6.5, color='#aaa', ha='left')
leg2 = [Line2D([0],[0], marker='D', color='w', markerfacecolor=C_OURS+'35',
               markeredgecolor=C_OURS, markersize=6, markeredgewidth=1.3, label='Ours'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor=C_3DGS+'35',
               markeredgecolor=C_3DGS, markersize=5, markeredgewidth=1.0, label='3DGS baseline'),
        Line2D([0],[0], marker='o', color='w', markerfacecolor=C_NYU+'35',
               markeredgecolor=C_NYU, markersize=5, markeredgewidth=1.0, label='NYU zero-shot')]
ax2.legend(handles=leg2, loc='center left', bbox_to_anchor=(0.01, 0.45),
           frameon=True, fancybox=False, edgecolor='#eee', fontsize=7.5)
fig2.tight_layout()
fig2.savefig(eval_dir / 'fig_params_vs_psnr.png', dpi=300, bbox_inches='tight', facecolor='white')
fig2.savefig(eval_dir / 'fig_params_vs_psnr.pdf', bbox_inches='tight', facecolor='white')
print('Saved fig_params_vs_psnr')
plt.show()


### 4.2 Training convergence


In [ ]:
# 4.2 Training convergence curves (val L1 over epochs)
import json as _json
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

logs = {}
log_files = {
    'UNet (ours)':     'depth_upres_8bit_ch32',
    'UNet+Normal':     'depth_upres_8bit_ch32_normal_w050_dsine',
    'EDSR-light':      'baseline_edsr',
    'SRResNet':        'baseline_srresnet',
    'UNet 16-bit':     'depth_upres_16bit_ch32',
}
for label, dirname in log_files.items():
    p = CKPT_ROOT / dirname / 'train_log.json'
    if p.exists():
        logs[label] = _json.loads(p.read_text())

style = {
    'UNet (ours)':  {'c': '#1a56db', 'ls': '-',  'lw': 1.8},
    'UNet+Normal':  {'c': '#1a56db', 'ls': '--', 'lw': 1.5},
    'UNet 16-bit':  {'c': '#e67e22', 'ls': '-',  'lw': 1.5},
    'EDSR-light':   {'c': '#4a9ec5', 'ls': '-',  'lw': 1.3},
    'SRResNet':     {'c': '#4a9ec5', 'ls': '--', 'lw': 1.3},
}

def _smooth(vals, w=5):
    if len(vals) < w: return vals
    kernel = np.ones(w) / w
    smoothed = np.convolve(vals, kernel, mode='valid')
    return list(vals[:w-1]) + list(smoothed)

fig, ax = plt.subplots(figsize=(7, 4.2))
for label, data in logs.items():
    epochs = [e['epoch'] for e in data]
    val_l1 = [e['val_l1'] for e in data]
    s = style.get(label, {'c': '#666', 'ls': '-', 'lw': 1.2})
    smoothed = _smooth(val_l1, w=5)
    best_val = min(val_l1)
    ax.plot(epochs[:len(smoothed)], smoothed, color=s['c'], linestyle=s['ls'],
            linewidth=s['lw'], label=f"{label} (best: {best_val:.5f})", alpha=0.9)

ax.set_xlabel('Epoch'); ax.set_ylabel(r'Val L$_1$')
ax.set_title('Training convergence', fontweight='bold', fontsize=11, pad=10)
ax.grid(True, linestyle=':', alpha=0.2)
ax.set_ylim(bottom=0.0018, top=0.005)
ax.legend(frameon=True, fancybox=False, edgecolor='#eee', fontsize=7.5, loc='upper right')

fig.tight_layout()
eval_dir = PROJECT_ROOT / 'eval'
fig.savefig(eval_dir / 'fig_convergence.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(eval_dir / 'fig_convergence.pdf', bbox_inches='tight', facecolor='white')
print('Saved fig_convergence')
plt.show()


### 4.3 Degradation cross-test heatmap


In [ ]:
# 4.3 Degradation cross-test heatmap (5x5 PSNR matrix)
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm', 'font.size': 10,
    'axes.spines.top': True, 'axes.spines.right': True,
})

eval_dir = PROJECT_ROOT / 'eval'
psnr_df = pd.read_csv(eval_dir / 'degradation_crosstest_PSNR.csv', index_col=0)
l1_df   = pd.read_csv(eval_dir / 'degradation_crosstest_L1.csv', index_col=0)

def _pretty(s):
    s = s.replace('train_', '').replace('test_', '')
    mapping = {'bicubic_noise': 'Bic+Noise', 'bicubic_q8': 'Bic+Q8',
               'render_q8': 'Render+Q8', 'bicubic': 'Bicubic', 'render': 'Render'}
    for k in sorted(mapping.keys(), key=len, reverse=True):
        if s == k: return mapping[k]
    return s

psnr_df.index   = [_pretty(s) for s in psnr_df.index]
psnr_df.columns = [_pretty(s) for s in psnr_df.columns]
l1_df.index     = [_pretty(s) for s in l1_df.index]
l1_df.columns   = [_pretty(s) for s in l1_df.columns]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={'wspace': 0.35})
vals_psnr = psnr_df.values
vals_l1   = l1_df.values * 1000

for ax_i, (vals, title, cmap) in enumerate([
    (vals_psnr, '(a) PSNR (dB)', 'Blues'),
    (vals_l1,   r'(b) L$_1$ ($\times 10^{-3}$)', 'Reds_r'),
]):
    ax = axes[ax_i]
    n = vals.shape[0]
    im = ax.imshow(vals, cmap=cmap, aspect='equal')
    labels = list(psnr_df.columns)
    ax.set_xticks(range(n)); ax.set_yticks(range(n))
    ax.set_xticklabels(labels, fontsize=8.5, rotation=25, ha='right')
    ax.set_yticklabels(list(psnr_df.index), fontsize=8.5)
    ax.set_xlabel('Test degradation', fontsize=10)
    ax.set_ylabel('Train degradation', fontsize=10)
    ax.set_title(title, fontweight='bold', fontsize=11, pad=10)
    fmt = '.1f' if ax_i == 0 else '.2f'
    mid = (vals.max() + vals.min()) / 2
    for i in range(n):
        for j in range(n):
            v = vals[i, j]
            bright = (v > mid) if ax_i == 0 else (v < mid)
            color = 'white' if bright else 'black'
            weight = 'bold' if i == j else 'normal'
            ax.text(j, i, f'{v:{fmt}}', ha='center', va='center',
                    fontsize=7.5, color=color, fontweight=weight)
    for i in range(n):
        ax.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, fill=False,
                     edgecolor='#333', linewidth=1.5, linestyle='--'))
    plt.colorbar(im, ax=ax, shrink=0.82, pad=0.02)

diag_psnr = np.diag(vals_psnr)
off_mask = ~np.eye(n, dtype=bool)
off_psnr = vals_psnr[off_mask]
gap = diag_psnr.mean() - off_psnr.mean()
fig.text(0.5, -0.01,
         f'Diagonal mean: {diag_psnr.mean():.1f} dB    Off-diagonal mean: {off_psnr.mean():.1f} dB    Gap: {gap:.1f} dB',
         ha='center', fontsize=9, color='#555')
fig.tight_layout()
fig.savefig(eval_dir / 'fig_crosstest_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(eval_dir / 'fig_crosstest_heatmap.pdf', bbox_inches='tight', facecolor='white')
print('Saved fig_crosstest_heatmap')
plt.show()


### 4.4 Qualitative comparison


In [ ]:
# 4.4 Qualitative comparison — zoomed crops + error maps
import matplotlib.pyplot as plt
import numpy as np
import torch, cv2
from PIL import Image
from pathlib import Path
from train_depth_upres import DepthUpResUNet

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

eval_dir = PROJECT_ROOT / 'eval'
DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

val_samples = sorted([p.stem for p in (DATASET_DIR / 'val' / 'depth').glob('*.png')])
PICK = [val_samples[0], val_samples[len(val_samples)//2]]

model = DepthUpResUNet(base_ch=32, predict_normal=False).to(device)
ckpt = torch.load(CKPT_ROOT / 'depth_upres_8bit_ch32' / 'best.pt', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model'], strict=False)
model.eval()

CY, CX, CS = 384, 384, 256

rows_data = []
for sample_name in PICK:
    hr_pil = Image.open(DATASET_DIR / 'val' / 'depth' / f'{sample_name}.png')
    hr = np.asarray(hr_pil).astype(np.float32)
    hr = hr / 65535.0 if hr.max() > 255 else hr / 255.0
    lr_pil = Image.open(DATASET_DIR / 'val' / 'depth_lr_8bit' / f'{sample_name}.png')
    lr = np.asarray(lr_pil).astype(np.float32) / 255.0
    bic = cv2.resize(lr, (1024, 1024), interpolation=cv2.INTER_CUBIC).clip(0, 1)
    lr_t = torch.from_numpy(lr).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(lr_t).clamp(0, 1).cpu().numpy()[0, 0]
    mask_path = DATASET_DIR / 'val' / 'mask' / f'{sample_name}.png'
    mask = (np.array(Image.open(mask_path).convert('L')) > 127).astype(np.float32) if mask_path.exists() else np.ones_like(hr)
    hr_c = hr[CY:CY+CS, CX:CX+CS]; bic_c = bic[CY:CY+CS, CX:CX+CS]
    pred_c = pred[CY:CY+CS, CX:CX+CS]; mask_c = mask[CY:CY+CS, CX:CX+CS]
    err_bic = np.abs(bic_c - hr_c) * mask_c
    err_pred = np.abs(pred_c - hr_c) * mask_c
    l1_bic = err_bic.sum() / mask_c.sum()
    l1_pred = err_pred.sum() / mask_c.sum()
    rows_data.append({'name': sample_name, 'hr': hr_c, 'bic': bic_c, 'pred': pred_c,
                      'err_bic': err_bic, 'err_pred': err_pred, 'l1_bic': l1_bic, 'l1_pred': l1_pred})

del model; torch.cuda.empty_cache()

fig, axes = plt.subplots(len(PICK), 5, figsize=(13, 5.5))
col_titles = ['HR (ground truth)', 'Bicubic', 'UNet (ours)', '|Error| Bicubic', '|Error| UNet']

for row_i, rd in enumerate(rows_data):
    err_vmax = max(rd['err_bic'].mean() * 10, 0.005)
    images = [rd['hr'], rd['bic'], rd['pred'], rd['err_bic'], rd['err_pred']]
    for col_i, img in enumerate(images):
        ax = axes[row_i, col_i]
        if col_i < 3:
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        else:
            im = ax.imshow(img, cmap='inferno', vmin=0, vmax=err_vmax)
            l1_val = rd['l1_bic'] if col_i == 3 else rd['l1_pred']
            ax.text(0.5, 0.95, f'L1 = {l1_val:.5f}', transform=ax.transAxes,
                    ha='center', va='top', fontsize=7.5, color='white',
                    bbox=dict(boxstyle='round,pad=0.25', fc='black', alpha=0.65, lw=0))
        ax.set_xticks([]); ax.set_yticks([])
        if row_i == 0:
            ax.set_title(col_titles[col_i], fontsize=9.5, pad=5)
    plt.colorbar(im, ax=axes[row_i, 4], fraction=0.046, pad=0.04)

fig.suptitle(f'Qualitative comparison (center crop {CS}$\\times${CS})', fontsize=12, fontweight='bold', y=1.01)
fig.tight_layout()
fig.savefig(eval_dir / 'fig_visual_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(eval_dir / 'fig_visual_comparison.pdf', bbox_inches='tight', facecolor='white')
print('Saved fig_visual_comparison')
plt.show()


### 4.5 Normal supervision ablation


In [ ]:
# 4.5 Normal supervision ablation — bar chart
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Data from project records:
# - No normal: UNet baseline, val_l1=0.00232
# - 3DGS normals (bilateral): plateaued at ~0.0037 (failed)
# - DSINE pseudo-GT: val_l1=0.00228 (matches/beats baseline)
configs = [
    ('No normal\n(baseline)',      0.00232, '#4a9ec5'),
    ('3DGS normals\n(bilateral)',  0.00370, '#dc2626'),
    ('DSINE\npseudo-GT',           0.00228, '#1a56db'),
]
labels, vals, cols = zip(*configs)

fig, ax = plt.subplots(figsize=(5, 4))
x = np.arange(len(labels))
bars = ax.bar(x, vals, width=0.55, color=[c+'30' for c in cols],
              edgecolor=cols, linewidth=1.3)

ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel(r'Val L$_1$ $\downarrow$', fontsize=11)
ax.set_title('Normal supervision ablation', fontweight='bold', fontsize=11, pad=10)
ax.grid(axis='y', linestyle=':', alpha=0.25)
ax.set_ylim(0, 0.005)

for i, (bar, v) in enumerate(zip(bars, vals)):
    color = cols[i]
    weight = 'bold' if v == min(vals) else 'normal'
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.00008, f'{v:.5f}',
            ha='center', va='bottom', fontsize=8.5, fontweight=weight, color=color)

# Annotation: 3DGS normals fail
ax.annotate('per-splat aliasing\n(unusable)', xy=(1, 0.00370),
            xytext=(1.6, 0.0044), fontsize=7.5, color='#dc2626',
            arrowprops=dict(arrowstyle='->', color='#dc2626', lw=0.8),
            ha='center')

fig.tight_layout()
eval_dir = PROJECT_ROOT / 'eval'
fig.savefig(eval_dir / 'fig_normal_ablation.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(eval_dir / 'fig_normal_ablation.pdf', bbox_inches='tight', facecolor='white')
print('Saved fig_normal_ablation')
plt.show()


### 4.6 Quantization gap (8-bit vs 16-bit)


In [ ]:
# 4.6 Quantization gap — 8-bit vs 16-bit input ablation
import json as _json
import matplotlib.pyplot as plt
import numpy as np

matplotlib.rcParams.update({
    'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})

# Load logs
log_8  = _json.loads((CKPT_ROOT / 'depth_upres_8bit_ch32' / 'train_log.json').read_text())
log_16 = _json.loads((CKPT_ROOT / 'depth_upres_16bit_ch32' / 'train_log.json').read_text())

best_8  = min(log_8,  key=lambda e: e['val_l1'])
best_16 = min(log_16, key=lambda e: e['val_l1'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), gridspec_kw={'wspace': 0.35})

# (a) Convergence comparison
C8 = '#1a56db'; C16 = '#e67e22'
ax1.plot([e['epoch'] for e in log_8],  [e['val_l1'] for e in log_8],
         color=C8, lw=1.8, label=f'8-bit input (best={best_8["val_l1"]:.5f})')
ax1.plot([e['epoch'] for e in log_16], [e['val_l1'] for e in log_16],
         color=C16, lw=1.8, ls='--', label=f'16-bit input (best={best_16["val_l1"]:.5f})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel(r'Val L$_1$')
ax1.set_title('(a) Convergence: 8-bit vs 16-bit', fontweight='bold', fontsize=10, pad=8)
ax1.legend(frameon=True, fancybox=False, edgecolor='#eee', fontsize=8)
ax1.grid(True, linestyle=':', alpha=0.2)
ax1.set_ylim(0.0015, 0.006)

# (b) Bar comparison
labels = ['8-bit\n(SR + dequant.)', '16-bit\n(SR only)']
vals = [best_8['val_l1'], best_16['val_l1']]
cols = [C8, C16]
bars = ax2.bar([0, 1], vals, width=0.5, color=[c+'30' for c in cols],
               edgecolor=cols, linewidth=1.3)
ax2.set_xticks([0, 1]); ax2.set_xticklabels(labels, fontsize=9)
ax2.set_ylabel(r'Val L$_1$ $\downarrow$')
ax2.set_title('(b) Quantization gap', fontweight='bold', fontsize=10, pad=8)
ax2.set_ylim(0, max(vals) * 1.4)
ax2.grid(axis='y', linestyle=':', alpha=0.25)

for i, (bar, v) in enumerate(zip(bars, vals)):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.00005, f'{v:.5f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=cols[i])

gap = best_8['val_l1'] - best_16['val_l1']
ax2.annotate(f'gap: {gap:.5f}\n({gap/best_16["val_l1"]*100:.1f}%)',
             xy=(0.5, max(vals)), xytext=(0.5, max(vals)*1.25),
             ha='center', fontsize=8, color='#666',
             arrowprops=dict(arrowstyle='-[', color='#999', lw=0.8))

fig.tight_layout()
eval_dir = PROJECT_ROOT / 'eval'
fig.savefig(eval_dir / 'fig_quantization_gap.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(eval_dir / 'fig_quantization_gap.pdf', bbox_inches='tight', facecolor='white')
print('Saved fig_quantization_gap')
plt.show()
